# EN3150 Assignment 03: Train & evaluate everything (Kaggle / Colab)

Runs every experiment the assignment needs **on one GPU**, so the *time per epoch* column is comparable across models.

| Assignment part | Cells |
|---|---|
| 1. Data prep (64×64, 70/15/15) | Step 2 |
| 3. Optimizer comparison (Adam vs SGD vs SGD+Momentum) | Step 4 |
| 4. Model A & B, 20 epochs, loss curves, test metrics, A-vs-B table | Steps 3, 6, 7 |
| 5. Fine-tune MobileNetV2 + EfficientNet-B0, params + size (MB) | Step 5 |
| 6. Model B vs SOTA comparison | Step 7 |

**Before running:**
- **Kaggle:** right sidebar → *Accelerator* = **GPU T4 x2** (or P100), *Internet* = **On**. Then use **Save Version → Save & Run All (Commit)** so it runs in the background, and download from the *Output* tab.
- **Colab:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Keep the tab open; Step 8 saves results to Google Drive.

## Step 0: Settings

In [ ]:
REPO_URL = "https://github.com/rash200319/CNN-for-image-classification.git"
# Private repo? use: "https://<YOUR_GITHUB_TOKEN>@github.com/rash200319/CNN-for-image-classification.git"

EPOCHS_CUSTOM = 20   # assignment: at least 20 epochs for Model A and Model B
EPOCHS_SOTA   = 5    # fine-tuning epochs for the pretrained models
RUN_MODEL_A_OPTIMIZER_SWEEP = False  # True = also compare SGD / SGD+Momentum for Model A (optional, +~2 runs)

## Step 1: Detect platform, check GPU, clone the repo

In [ ]:
import os, sys, subprocess, pathlib

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules
WORK_DIR  = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path("/content")
REPO_DIR  = WORK_DIR / "CNN-for-image-classification"
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Other/local")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU, enable it in the notebook settings, otherwise training is very slow.")

import socket, shutil
def has_internet(host="github.com"):
    try:
        socket.gethostbyname(host); return True
    except OSError:
        return False

HAS_INTERNET = has_internet()
print("Internet:", HAS_INTERNET)

if HAS_INTERNET:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
elif not REPO_DIR.exists():
    # Offline fallback: repo uploaded as a Kaggle dataset (any folder under /kaggle/input containing src/train.py)
    found = next((p.parent.parent for p in pathlib.Path("/kaggle/input").rglob("src/train.py")), None)
    if found is None:
        raise RuntimeError(
            "No internet. On Kaggle: right sidebar -> Session options -> Internet = ON "
            "(needs a phone-verified account, Settings -> Phone verification), then Restart & Run All.\n"
            "Or upload the repo folder as a Kaggle dataset and attach it with '+ Add Input'.")
    shutil.copytree(found, REPO_DIR, ignore=shutil.ignore_patterns(".git", "__pycache__"))
    print("Copied repo from", found)

os.chdir(REPO_DIR)
print("Working in:", os.getcwd())

In [ ]:
# torch / torchvision / sklearn / pandas / matplotlib are preinstalled on Kaggle & Colab
!pip install -q tqdm

## Step 2: Get the EuroSAT RGB dataset
The code expects the images at `archive/EuroSAT/<ClassName>/*.jpg`. This cell:
1. looks for an already-attached copy (a Kaggle dataset under `/kaggle/input`, or a folder in your Google Drive), otherwise
2. downloads EuroSAT RGB (~90 MB) with `torchvision`,

then links it to `archive/EuroSAT`.

In [ ]:
from pathlib import Path

def find_eurosat(root):
    root = Path(root)
    if not root.exists():
        return None
    for p in root.rglob("AnnualCrop"):
        if p.is_dir() and (p.parent / "SeaLake").is_dir() and any(p.glob("*.jpg")):
            return p.parent
    return None

search_roots = ["/kaggle/input"]
if ON_COLAB:
    # Optional: put your EuroSAT_RGB folder in Google Drive and it will be used instead of downloading
    from google.colab import drive
    drive.mount("/content/drive")
    search_roots.append("/content/drive/MyDrive")

data_root = None
for r in search_roots:
    data_root = find_eurosat(r)
    if data_root:
        break

if data_root is None:
    print("No local copy found, downloading EuroSAT RGB with torchvision ...")
    from torchvision.datasets import EuroSAT
    EuroSAT(root="/tmp/eurosat_dl", download=True)
    data_root = find_eurosat("/tmp/eurosat_dl")

assert data_root is not None, "EuroSAT not found / download failed"
print("Dataset folder:", data_root)

link = REPO_DIR / "archive" / "EuroSAT"
link.parent.mkdir(exist_ok=True)
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(data_root, target_is_directory=True)

classes = sorted(d.name for d in link.iterdir() if d.is_dir())
n_imgs = sum(len(list((link / c).glob("*.jpg"))) for c in classes)
print(f"{len(classes)} classes, {n_imgs} images:", classes)
assert len(classes) == 10 and n_imgs == 27000

In [ ]:
# Sanity check: split sizes + parameter counts (Model B must be <= 100k)
from src.data.dataset import get_dataloaders
from src.models.model_a import ModelA
from src.models.model_b import ModelB
from src.utils import count_parameters

tr, va, te, cls = get_dataloaders()
print(f"train={len(tr.dataset)}  val={len(va.dataset)}  test={len(te.dataset)}")
print(f"Model A params: {count_parameters(ModelA(10)):,}")
print(f"Model B params: {count_parameters(ModelB(10)):,}  (limit 100,000)")

## Step 3: Train Model A and Model B with Adam (Part 4)

In [ ]:
!python -m src.train --model a --optimizer adam --lr 1e-3 --epochs {EPOCHS_CUSTOM} --run-name final_a_adam

In [ ]:
!python -m src.train --model b --optimizer adam --lr 1e-3 --epochs {EPOCHS_CUSTOM} --run-name modelb_adam

## Step 4: Optimizer comparison on Model B: plain SGD vs SGD + Momentum (Part 3)

In [ ]:
!python -m src.train --model b --optimizer sgd          --lr 1e-2                --epochs {EPOCHS_CUSTOM} --run-name modelb_sgd
!python -m src.train --model b --optimizer sgd_momentum --lr 1e-2 --momentum 0.9 --epochs {EPOCHS_CUSTOM} --run-name modelb_sgd_momentum

In [ ]:
# Optional: same comparison for Model A
if RUN_MODEL_A_OPTIMIZER_SWEEP:
    !python -m src.train --model a --optimizer sgd          --lr 1e-2                --epochs {EPOCHS_CUSTOM} --run-name modela_sgd
    !python -m src.train --model a --optimizer sgd_momentum --lr 1e-2 --momentum 0.9 --epochs {EPOCHS_CUSTOM} --run-name modela_sgd_momentum

## Step 5: Fine-tune the two lightweight pretrained models (Part 5)

Both SOTA compliance runs use the assignment maximum of 64x64. The historical 128x128 MobileNetV2 result is not regenerated here and must be treated only as an optional resolution study.


In [ ]:
!python -m src.train --model mobilenet_v2    --optimizer adam --lr 1e-4 --img-size 64 --epochs {EPOCHS_SOTA} --run-name final_mobilenet_v2
!python -m src.train --model efficientnet_b0 --optimizer adam --lr 1e-4 --img-size 64  --epochs {EPOCHS_SOTA} --run-name final_efficientnet_b0

## Step 6: Evaluate on the test set and plot loss curves

In [ ]:
RUNS = {  # run_name: (model flag, img size)
    "final_a_adam":          ("a", 64),
    "modelb_adam":           ("b", 64),
    "modelb_sgd":            ("b", 64),
    "modelb_sgd_momentum":   ("b", 64),
    "final_mobilenet_v2":    ("mobilenet_v2", 64),
    "final_efficientnet_b0": ("efficientnet_b0", 64),
}
if RUN_MODEL_A_OPTIMIZER_SWEEP:
    RUNS.update({"modela_sgd": ("a", 64), "modela_sgd_momentum": ("a", 64)})

for run, (m, size) in RUNS.items():
    print(f"\n===== {run} =====")
    !python -m src.evaluate --model {m} --checkpoint outputs/{run}/best_model.pt --img-size {size}
    !python -m src.plot_history --run-dir outputs/{run}

In [ ]:
# Overlay plots: optimizer comparison (Part 3) and Model A vs B (Part 4)
!python -m src.plot_history --run-dir outputs/modelb_adam --run-dir outputs/modelb_sgd --run-dir outputs/modelb_sgd_momentum --out outputs/modelb_optimizer_comparison.png
!python -m src.plot_history --run-dir outputs/final_a_adam --run-dir outputs/modelb_adam --out outputs/modelA_vs_modelB_loss.png
if RUN_MODEL_A_OPTIMIZER_SWEEP:
    !python -m src.plot_history --run-dir outputs/final_a_adam --run-dir outputs/modela_sgd --run-dir outputs/modela_sgd_momentum --out outputs/modela_optimizer_comparison.png

## Step 7: Results tables for the report (Parts 3, 4, 5, 6)

In [ ]:
import json, pandas as pd
from IPython.display import display, Image, Markdown

rows = []
for run in RUNS:
    d = Path("outputs") / run
    m = json.load(open(d / "metrics.json"))
    h = pd.read_csv(d / "history.csv")
    rows.append({
        "run": run,
        "params": m["trainable_params"],
        "size_KB": round(m["checkpoint_size_kb"], 1),
        "size_MB": round(m["checkpoint_size_kb"] / 1024, 3),
        "epochs": len(h),
        "median_s_per_epoch": round(h["epoch_time_sec"].median(), 2),
        "best_val_acc_%": round(100 * h["val_acc"].max(), 2),
        "test_acc_%": round(100 * m["test_accuracy"], 2),
        "precision_macro": round(m["precision_macro"], 4),
        "recall_macro": round(m["recall_macro"], 4),
        "f1_macro": round(m["f1_macro"], 4),
    })
df = pd.DataFrame(rows).set_index("run")
df.to_csv("outputs/all_results.csv")

display(Markdown("### Part 3: Optimizer comparison (Model B)"))
display(df.loc[["modelb_adam", "modelb_sgd", "modelb_sgd_momentum"]])
display(Markdown("### Part 4: Model A vs Model B"))
display(df.loc[["final_a_adam", "modelb_adam"], ["params", "size_KB", "median_s_per_epoch", "test_acc_%"]])
display(Markdown("### Parts 5 & 6: Model B vs lightweight SOTA"))
display(df.loc[["modelb_adam", "final_mobilenet_v2", "final_efficientnet_b0"],
               ["params", "size_MB", "median_s_per_epoch", "test_acc_%", "precision_macro", "recall_macro"]])

print("GPU used:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print(df.to_markdown())

In [ ]:
# Per-class precision / recall (for the report's confusion-matrix discussion)
for run in ["final_a_adam", "modelb_adam", "final_mobilenet_v2", "final_efficientnet_b0"]:
    pc = json.load(open(f"outputs/{run}/metrics.json"))["per_class"]
    display(Markdown(f"**{run}**"))
    display(pd.DataFrame(pc).T.round(4))

In [ ]:
# Show all figures inline
for run in RUNS:
    display(Markdown(f"### {run}"))
    for f in ["loss_curve.png", "confusion_matrix.png"]:
        p = Path("outputs") / run / f
        if p.exists():
            display(Image(filename=str(p), width=520))
for f in ["modelb_optimizer_comparison.png", "modelA_vs_modelB_loss.png", "modela_optimizer_comparison.png"]:
    p = Path("outputs") / f
    if p.exists():
        display(Markdown(f"### {f}")); display(Image(filename=str(p), width=700))

## Step 8: Save / download the results
- **Kaggle:** `results.zip` is written to `/kaggle/working` and appears in the notebook's **Output** tab after the committed run finishes.
- **Colab:** it's copied to `MyDrive/EN3150_A03_results.zip` and also offered as a browser download.

Unzip it into your local repo so that `outputs/` and the figures can be used in `report/main.tex`.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(WORK_DIR / "results"), "zip", root_dir=REPO_DIR, base_dir="outputs")
print("Created", zip_path)

if ON_COLAB:
    shutil.copy(zip_path, "/content/drive/MyDrive/EN3150_A03_results.zip")
    print("Copied to Google Drive: MyDrive/EN3150_A03_results.zip")
    from google.colab import files
    files.download(zip_path)